In [1]:
import pandas as pd 
import numpy as np
import tensorflow as tf


In [2]:
df = pd.read_csv("./resources/diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.corr()['Outcome'].sort_values(ascending=False)

Outcome                     1.000000
Glucose                     0.466581
BMI                         0.292695
Age                         0.238356
Pregnancies                 0.221898
DiabetesPedigreeFunction    0.173844
Insulin                     0.130548
SkinThickness               0.074752
BloodPressure               0.065068
Name: Outcome, dtype: float64

In [8]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [11]:
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense


In [15]:
model = Sequential()
model.add(Dense(32 , activation = 'relu' , input_dim = 8))
model.add(Dense(1 , activation = 'sigmoid'))

model.compile(optimizer='adam' , loss = 'binary_crossentropy' , metrics = ['accuracy'])


c:\Users\mrsan\OneDrive\Desktop\code\python\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.fit(X_train , y_train , batch_size= 32 , epochs = 100 , validation_data=(X_test , y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6531 - loss: 15.8869 - val_accuracy: 0.6429 - val_loss: 9.4693
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6433 - loss: 6.0289 - val_accuracy: 0.6169 - val_loss: 3.8234
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5912 - loss: 3.1598 - val_accuracy: 0.5909 - val_loss: 2.0717
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5928 - loss: 1.8845 - val_accuracy: 0.5325 - val_loss: 1.5852
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5831 - loss: 1.3975 - val_accuracy: 0.5455 - val_loss: 1.2645
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6205 - loss: 1.0871 - val_accuracy: 0.5649 - val_loss: 1.1094
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6287 - loss: 0.9507 - val_accuracy: 0.6234 - val_loss: 0.9711
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6629 - loss: 0.8336 - val_accuracy: 0.6558 -

In [18]:
import pandas as pd
import tensorflow as tf
import keras_tuner as kt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [19]:
def build_model(hp):
    model = Sequential()

    # tune no. of hidden layers 

    num_layers = hp.Int(
        "num_layers",
        min_value = 1,
        max_value = 5
    )

    for i in range(num_layers):

        units = hp.Int(
            f"units_{i}",
            min_value = 16,
            max_value = 128,
            step = 8
        )

        if i == 0:
            model.add(Dense(
                units,
                activation='relu',
                input_shape=(8,)
            ))

        else:
            model.add(Dense(
                units,
                activation='relu'
            ))
        
        dropout_rate = hp.Choice(
            f'dropout_{i}',
            values=[0.1,0.2,0.3,0.4,0.5]
        )
    
    model.add(Dense(1,
                    activation='sigmoid'
                    ))
    optimizer = hp.Choice(
        'optimizer',
        values=[
            'adam',
            'sgd',
            'rmsprop'
        ]
    )


    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [20]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=10,

    executions_per_trial=1,

    directory="keras_tuner",

    project_name="diabetes_project"
)

c:\Users\mrsan\OneDrive\Desktop\code\python\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True
)

In [23]:
tuner.search(

    X_train,

    y_train,

    epochs=50,

    validation_split=0.2,

    #callbacks=[early_stop]
)


Trial 10 Complete [00h 00m 05s]
val_accuracy: 0.7317073345184326

Best val_accuracy So Far: 0.7804877758026123
Total elapsed time: 00h 02m 30s


In [24]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print("Best Optimizer :", best_hp.get("optimizer"))

print("Best Number of Layers :", best_hp.get("num_layers"))

for i in range(best_hp.get("num_layers")):

    print(f"Units in Layer {i+1} :", best_hp.get(f"units_{i}"))

    print(f"Dropout {i+1} :", best_hp.get(f"dropout_{i}"))



Best Optimizer : adam
Best Number of Layers : 2
Units in Layer 1 : 48
Dropout 1 : 0.4
Units in Layer 2 : 128
Dropout 2 : 0.2


In [25]:
# ---------------------------------
# Get Best Model
# ---------------------------------

best_model = tuner.get_best_models(1)[0]


c:\Users\mrsan\OneDrive\Desktop\code\python\venv\lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:

# ---------------------------------
# Train Best Model Again
# ---------------------------------

history = best_model.fit(

    X_train,

    y_train,

    epochs=100,

    validation_split=0.2,

    # callbacks=[early_stop]
)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6619 - loss: 0.8647 - val_accuracy: 0.6911 - val_loss: 0.6206
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6802 - loss: 0.6911 - val_accuracy: 0.7073 - val_loss: 0.6704
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6884 - loss: 0.6408 - val_accuracy: 0.7154 - val_loss: 0.5456
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7108 - loss: 0.5591 - val_accuracy: 0.6667 - val_loss: 0.6264
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7556 - loss: 0.5370 - val_accuracy: 0.6992 - val_loss: 0.6574
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6864 - loss: 0.5831 - val_accuracy: 0.7398 - val_loss: 0.5390
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7332 - loss: 0.5312 - val_accuracy: 0.7480 - val_loss: 0.5394
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7332 - loss: 0.5026 - val_accuracy: 0.6423 - v

In [27]:
# ---------------------------------
# Evaluate Model
# ---------------------------------

loss, accuracy = best_model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy :", accuracy)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7013 - loss: 0.8182 
Test Accuracy : 0.701298713684082


In [28]:
# ---------------------------------
# Show Search Summary
# ---------------------------------

tuner.results_summary()

Results summary
Results in keras_tuner\diabetes_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 06 summary
Hyperparameters:
num_layers: 2
units_0: 48
dropout_0: 0.4
optimizer: adam
units_1: 128
dropout_1: 0.2
units_2: 104
dropout_2: 0.3
units_3: 40
dropout_3: 0.2
Score: 0.7804877758026123

Trial 07 summary
Hyperparameters:
num_layers: 1
units_0: 112
dropout_0: 0.3
optimizer: adam
units_1: 16
dropout_1: 0.4
units_2: 48
dropout_2: 0.5
units_3: 40
dropout_3: 0.3
Score: 0.7804877758026123

Trial 02 summary
Hyperparameters:
num_layers: 3
units_0: 80
dropout_0: 0.5
optimizer: rmsprop
units_1: 16
dropout_1: 0.1
units_2: 16
dropout_2: 0.1
Score: 0.7642276287078857
Traceback (most recent call last):
  File "c:\Users\mrsan\OneDrive\Desktop\code\python\venv\lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\mrsan\OneDrive\Desktop\code\p